In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Constants
num_epochs = 1000
batch_size = 32
learning_rate = .2
RANDOM_STATE = 1082
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set numpy random seed
np.random.seed(RANDOM_STATE)
# Set torch random seed
torch.manual_seed(RANDOM_STATE)
# If using CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
# Load the dataset
try:
    df = pd.read_csv("../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv' was not found.")
    exit()

In [4]:
# Preprocess the dataset
# Clean up column
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Split features and label
X = df.drop(columns=['timestamp', 'label'])
y = df['label']

# Encode labels
from sklearn.preprocessing import LabelEncoder
if y.dtype == object or y.dtype.name == 'category':
    le = LabelEncoder()
    y = le.fit_transform(y)
    
# scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split benign and malicious
benign_mask = y == 0
X_benign = X_scaled[benign_mask]
y_benign = y[benign_mask]

# Training set: benign only
X_train, X_benign_val, y_train, y_benign_val = train_test_split(
    X_benign, y_benign, test_size=0.2, random_state=RANDOM_STATE
)

# Test set: all samples (benign and malicious)
X_test = X_scaled
y_test = y

# Convert to torch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

In [5]:
# Define Autoencoder
class Autoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dim=8):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, encoding_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

input_dim = X_train.shape[1]
encoding_dim = 8
autoencoder = Autoencoder(input_dim, encoding_dim)

In [6]:
# Training

criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=learning_rate)

train_loader = torch.utils.data.DataLoader(X_train_tensor, batch_size=batch_size, shuffle=True)

autoencoder.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        output = autoencoder(batch)
        loss = criterion(output, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch.size(0)
    epoch_loss /= len(X_train_tensor)
    if (epoch+1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.6f}")


Epoch 1/1000, Loss: 0.230426
Epoch 10/1000, Loss: 0.210063
Epoch 20/1000, Loss: 0.208565
Epoch 30/1000, Loss: 0.207717
Epoch 40/1000, Loss: 0.208669
Epoch 50/1000, Loss: 0.208145
Epoch 60/1000, Loss: 0.209377
Epoch 70/1000, Loss: 0.207994
Epoch 80/1000, Loss: 0.208708
Epoch 90/1000, Loss: 0.208788
Epoch 100/1000, Loss: 0.210175
Epoch 110/1000, Loss: 0.208068
Epoch 120/1000, Loss: 0.208418
Epoch 130/1000, Loss: 0.208192
Epoch 140/1000, Loss: 0.208442
Epoch 150/1000, Loss: 0.208704
Epoch 160/1000, Loss: 0.208811
Epoch 170/1000, Loss: 0.209021
Epoch 180/1000, Loss: 0.208871
Epoch 190/1000, Loss: 0.207728
Epoch 200/1000, Loss: 0.208040
Epoch 210/1000, Loss: 0.209567
Epoch 220/1000, Loss: 0.208620
Epoch 230/1000, Loss: 0.208829
Epoch 240/1000, Loss: 0.208372
Epoch 250/1000, Loss: 0.209866
Epoch 260/1000, Loss: 0.209699
Epoch 270/1000, Loss: 0.207365
Epoch 280/1000, Loss: 0.209219
Epoch 290/1000, Loss: 0.207515
Epoch 300/1000, Loss: 0.207739
Epoch 310/1000, Loss: 0.208494
Epoch 320/1000, Los

In [7]:
# Evaluation
autoencoder.eval()
with torch.no_grad():
    reconstructed = autoencoder(X_test_tensor)
    mse = torch.mean((X_test_tensor - reconstructed) ** 2, dim=1).cpu().numpy()

# Set threshold (95th percentile of benign errors from training data)
with torch.no_grad():
    recon_benign_val = autoencoder(torch.tensor(X_benign_val, dtype=torch.float32))
    mse_benign_val = torch.mean((torch.tensor(X_benign_val, dtype=torch.float32) - recon_benign_val) ** 2, dim=1).cpu().numpy()
threshold = np.percentile(mse_benign_val, 95)

# Predict: anomaly if error > threshold
y_pred = (mse > threshold).astype(int)

In [8]:
# Results
print(f"Autoencoder Threshold: {threshold:.4f}")
print(f"Model Accuracy on Test Set: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Autoencoder Threshold: 1.1137
Model Accuracy on Test Set: 96.44%
Confusion Matrix:
[[3003  121]
 [   8  490]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98      3124
           1       0.80      0.98      0.88       498

    accuracy                           0.96      3622
   macro avg       0.90      0.97      0.93      3622
weighted avg       0.97      0.96      0.97      3622

